# Modèle Hybride 
### Sentence Transformers et critères structurés

Dans ce modèle, nous allons combiner deux approches :
- la similarité sémantique (Sentence Transformers)
- les critères structurés du dataset (famille olfactive, caractère, saisons, moments de la journée, intensité), qui permettent de respecter les préférences précises du questionnaire

Ainsi chaque parfum reçoit 5 sous-scores entre 0 et 1, combinés dans un score final pondéré. 
Les 3 meilleurs parfums sont recommandés avec une explication.

In [22]:
# On commence par la lecture du fichier nettoyé
import pandas as pd
import numpy as np

df = pd.read_csv("../../datasets/clean/perfumes_clean.csv")
df.head()


,id,name,house,url,family,secondary_family,character,top_notes,heart_notes,base_notes,...,times_of_day,season_spring,season_summer,season_autumn,season_winter,time_day,time_evening,time_night,description,text_full
0,ysl-libre,Libre,Yves Saint Laurent,https://www.scento.com/perfumes/libre-yves-sai...,White Bloom,NaN,floral; creamy; fresh; citrusy,mandarin; neroli; lavender,orange blossom; jasmine sambac,musk; vanilla,...,day; evening,1,1,1,0,1,1,0,Mandarin and neroli brighten a creamy bloom of...,Mandarin and neroli brighten a creamy bloom of...
1,guerlain-shalimar,Shalimar,Guerlain,https://www.scento.com/perfumes/shalimar-eau-d...,Petal Garden,Velvet Sweet,floral; sweet; warm; powdery,floral notes; bergamot,iris; jasmine; rose,vanilla; opoponax; tonka bean,...,day; evening,1,1,1,0,1,1,0,Floral notes and a flash of bergamot sit above...,Floral notes and a flash of bergamot sit above...
2,tom-ford-oud-wood,Oud Wood,Tom Ford,https://www.scento.com/perfumes/oud-wood-tom-ford,Spice Route,NaN,warm; spicy; sweet; earthy,NaN,NaN,NaN,...,evening; night,0,0,1,1,0,1,1,Smooth woods and spice: cardamom and pink pepp...,Smooth woods and spice: cardamom and pink pepp...
3,creed-aventus,Aventus,Creed,https://www.scento.com/perfumes/aventus-creed,Orchard Fruit,Wood & Root,fruity; earthy; citrusy; floral,lemon; pink pepper; apple; bergamot; blackcurrant,pineapple; jasmine; patchouli,birch; ambroxan; cedarwood; oakmoss; musk,...,day; evening,1,1,1,0,1,1,0,Sparkling lemon and blackcurrant open onto pin...,Sparkling lemon and blackcurrant open onto pin...
4,mfk-baccarat-rouge-540,Baccarat Rouge 540,Maison Francis Kurkdjian,https://www.scento.com/perfumes/baccarat-rouge...,Wood & Root,White Bloom,woody; warm; floral; spicy,jasmine; saffron,cedarwood,ambergris,...,day; evening,1,0,1,1,1,1,0,"Jasmine and saffron open airy and golden, held...","Jasmine and saffron open airy and golden, held..."


Lors du nettoyage, les cases vides des colonnes texte (secondary_family, top_notes, heart_notes, base_notes) avaient été remplacées par une chaîne vide "". Mais à la lecture du CSV, pandas transforme les cases vides en NaN. On les remet donc à "".

Seule la colonne intensity garde ses 251 valeurs manquantes (concentration inconnue) : elles recevront une valeur neutre dans le score d’intensité.

In [23]:
colonnes_texte = df.select_dtypes(include=["object", "string"]).columns
df[colonnes_texte] = df[colonnes_texte].fillna("")

print("Valeurs manquantes restantes :")
print(df.isna().sum()[df.isna().sum() > 0])

Valeurs manquantes restantes :
intensity    251
dtype: int64


Chargement du modèle Sentence Transformers 

On utilise le modèle all-MiniLM-L6-v2. C’est un modèle anglais, adapté à notre dataset dont toutes les descriptions sont en anglais : l’utilisateur devra donc écrire sa demande en anglais. 
Ce modèle est déjà pré-entraîné : contrairement à Word2Vec, on n’a pas besoin de l’entraîner, ni de tokéniser ou de retirer les stop words. Il est aussi léger et rapide, ce qui convient à une application interactive.

In [24]:

%pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [25]:
import sys
print(sys.executable)

from sentence_transformers import SentenceTransformer
modele = SentenceTransformer('all-MiniLM-L6-v2')

c:\Users\JL\anaconda3\envs\parfum\python.exe


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2861.30it/s]


Création des vecteurs de parfums

On encode deux colonnes pour chaque parfum :

- text_full qui contient les informations du parfum et qui permettra de comparer la description de l'utilisateur

- character qui contient 4 ajdectifs et qui permettra de comparer les émotions de l'utilisateur.
On les transforme en phrase car le modèle comprend mieux une phrase qu'une liste de mots.

In [26]:
df["character_phrase"] = "A " + df["character"].str.replace("; ", ", ") + " scent."
print(df["character_phrase"].iloc[0])

vecteurs_texte = modele.encode(df["text_full"].tolist(), show_progress_bar=True)
vecteurs_character = modele.encode(df["character_phrase"].tolist(), show_progress_bar=True)

print("Dimensions des vecteurs texte :", vecteurs_texte.shape)
print("Dimensions des vecteurs character :", vecteurs_character.shape)

A floral, creamy, fresh, citrusy scent.


Batches: 100%|██████████| 47/47 [00:03<00:00, 14.09it/s]

Dimensions des vecteurs texte : (1479, 384)
Dimensions des vecteurs character : (1479, 384)


On obtient ansi deux tableau de 1479 parfums, un pour text_full et l'autre pour character.
Chaque parfum y est représenté par un vecteur de 384 nombres, qui traduit le sens de son texte. 
Cette taille de 384 est fixée par le modèle utilisé.

Test 1 : similarité sémantique seule

Avant de construire le modèle hybride, on vérifie que Sentence Transformers comprend bien une demande en langage naturel. On encode une phrase en anglais et on la compare à tous les parfums avec la similarité cosinus.

In [27]:
from sklearn.metrics.pairwise import cosine_similarity

demande = "a fresh and light citrus perfume for summer"
vecteur_demande = modele.encode([demande])

scores = cosine_similarity(vecteur_demande, vecteurs_texte)[0]

resultats = df[["name", "house", "family", "character", "seasons"]].copy()
resultats["similarite"] = scores
resultats.sort_values(by="similarite", ascending=False).head(5)

,name,house,family,character,seasons,similarite
269,Cefiro,Floris London,Sunlit Citrus,citrusy; fresh; spicy; woody,spring; summer; autumn,0.711477
563,Cologne du Maghreb,Tauer Perfumes,Sunlit Citrus,floral; citrusy; fresh; herbal,spring; summer; autumn,0.689390
623,Colonia,Acqua di Parma,Green Leaf,fresh; herbal; citrusy; earthy,spring; summer,0.671394
226,Danger Pour Femme,Roja Parfums,White Bloom,floral; citrusy; creamy; powdery,spring; summer; autumn,0.663497
92,Black Aoud,Montale,Wood & Root,citrusy; woody; floral; earthy,spring; summer; autumn,0.655798


On retrouve bien les parfums de notre demande comme le Green Leaf ou le Sunlit Citrus.

Mais on voit aussi la limite de la similarité seule : Black Aoud (famille Wood & Root) apparaît en 5ᵉ position, alors qu’un parfum boisé à l’oud ne correspond pas à « fresh and light ». La similarité mesure une proximité de sens global : elle ne garantit pas le respect de critères précis. 
De plus, les négations sont mal comprises, et l’intensité ne figure pas dans text_full, donc la similarité ne peut pas en tenir compte. C’est l’intérêt du modèle hybride.

On teste maintenant une entrée utilisateur (questionnaire).

On simule les réponses d’un utilisateur, telles qu’elles seront collectées par le questionnaire Streamlit :

- deux réponses en texte libre (description du parfum recherché et émotions souhaitées) 
- des familles cochées, des saisons et des moments choisis ;
- une intensité sur une échelle de 1 à 5 
- des notes détestées

In [28]:
profil = {
    "description": "I am looking for a warm and comforting perfume, woody and slightly spicy, for winter evenings",
    "emotions": "cosy, reassuring, enveloping",
    "familles": ["Wood & Root", "Spice Route"],
    "saisons": ["autumn", "winter"],
    "moments": ["evening"],
    "intensite": 3,
    "notes_detestees": ["vanilla"],
}

On commence par retirer les parfums qui contiennet une note détestée car si non, le modèle pourrait conseillé un parfum très proche au lieu de l'exclure.

In [29]:
contient_note_detestee = pd.Series(False, index=df.index)
for note in profil["notes_detestees"]:
    contient_note_detestee = contient_note_detestee | df["notes"].str.contains(note,case=False)

candidats = df[~contient_note_detestee].copy()

print("Parfum avant filtre :", len(df))
print("Parfum après filtre :", len(candidats))


Parfum avant filtre : 1479
Parfum après filtre : 948


Ensuite, nous allons donner 5 sous scores entre 0 et 1 à nos parfums, un par critère.
Lorsque l'utilisateur ne répond pas, il reçoit un score de 0,5 (valeur neutre) qui ne pénalise/favorise pas le parfum. 

In [30]:
def normaliser(scores):
    ecart = scores.max() - scores.min()
    if ecart == 0:  # tous les scores identiques : on renvoie une valeur neutre
        return np.full(len(scores), 0.5)
    return (scores - scores.min()) / ecart

S1 : similarité sémantique de la description. C’est le cœur sémantique du modèle : on compare la description libre de l’utilisateur au text_full de chaque parfum.

In [31]:
vecteur_description = modele.encode([profil["description"]])
cosinus_description = cosine_similarity(vecteur_description, vecteurs_texte[candidats.index])[0]

candidats["s_semantique"] = normaliser(cosinus_description)
candidats["cosinus_brut"] = cosinus_description  # gardé pour l'explication

candidats[["name", "family", "cosinus_brut", "s_semantique"]].sort_values(by="s_semantique", ascending=False).head(5)

,name,family,cosinus_brut,s_semantique
563,Cologne du Maghreb,Sunlit Citrus,0.620920,1.000000
623,Colonia,Green Leaf,0.616562,0.989679
156,Sunday Cologne,Wood & Root,0.609978,0.974081
113,Toy Boy,Petal Garden,0.608624,0.970873
732,Bucephalus No. XII,Spice Route,0.606610,0.966102


S2 : famille olfactive. Le parfum obtient 1 si sa famille principale fait partie des familles cochées, 0,5 si c’est sa famille secondaire, et 0 sinon.

In [32]:
famille_principale = candidats["family"].isin(profil["familles"])
famille_secondaire = candidats["secondary_family"].isin(profil["familles"])

candidats["s_famille"] = np.where(famille_principale, 1.0, np.where(famille_secondaire, 0.5, 0.0))

candidats["s_famille"].value_counts()

s_famille
0.0    521
1.0    268
0.5    159
Name: count, dtype: int64

S3 : affinité émotionnelle. On compare les émotions décrites par l’utilisateur au caractère du parfum. C’est aussi une comparaison sémantique : par exemple cosy doit se rapprocher de warm, creamy ou sweet, sans être le même mot.

In [33]:
vecteur_emotions = modele.encode([profil["emotions"]])
cosinus_emotions = cosine_similarity(vecteur_emotions, vecteurs_character[candidats.index])[0]

candidats["s_emotion"] = normaliser(cosinus_emotions)

candidats[["name", "character", "s_emotion"]].sort_values(by="s_emotion", ascending=False).head(5)

,name,character,s_emotion
896,Victory Man,fruity; spicy; earthy; fresh,1.000000
1144,"Seductive Homme,",earthy; spicy; warm; powdery,0.958743
156,Sunday Cologne,earthy; warm; spicy; fresh,0.933587
152,Supremacy Not Only Intense,earthy; fruity; fresh; herbal,0.933555
837,Odyssey Wild One,spicy; earthy; warm; fresh,0.912401


S4 : saison et moment de la journée
On utilise les colonnes binaires créées lors du nettoyage. 
Si l’utilisateur choisit l’automne et l’hiver, un parfum qui couvre les deux saisons obtient 1, un parfum qui n’en couvre qu’une obtient 0,5.

La saison compte pour 70 % et le moment pour 30 %, 93 % des parfums sont portables le jour et le soir.
Le moment de la journée différencie peu les parfums

In [34]:
score_saison = candidats[[f"season_{s}" for s in profil["saisons"]]].mean(axis=1)
score_moment = candidats[[f"time_{m}" for m in profil["moments"]]].mean(axis=1)

candidats["s_saison"] = 0.7 * score_saison + 0.3 * score_moment

candidats["s_saison"].value_counts().sort_index()

s_saison
0.00     37
0.65    594
1.00    317
Name: count, dtype: int64

S5 : intensité. On compare l’intensité souhaitée (1 à 5) à celle du parfum, déduite de sa concentration lors du nettoyage. Un écart nul donne 1, un écart maximal (4) donne 0. 
Les parfums d’intensité inconnue reçoivent 0,5.

In [35]:
ecart = (candidats["intensity"] - profil["intensite"]).abs() / 4
candidats["s_intensite"] = (1 - ecart).fillna(0.5)

candidats[["name", "concentration", "intensity", "s_intensite"]].head(5)

,name,concentration,intensity,s_intensite
2,Oud Wood,eau-de-parfum,3.0,1.0
3,Aventus,eau-de-parfum,3.0,1.0
4,Baccarat Rouge 540,eau-de-parfum,3.0,1.0
5,Santal 33,eau-de-parfum,3.0,1.0
8,J'adore,eau-de-parfum,3.0,1.0


| Critère | Poids | Justification |
|---|---|---|
| Description (S1) | 0,45 | entrée la plus riche, cœur de l'analyse sémantique |
| Famille (S2) | 0,20 | premier critère de choix d'un parfum |
| Émotions (S3) | 0,15 | dimension centrale du thème des parfums |
| Saison et moment (S4) | 0,12 | critère d'ajustement |
| Intensité (S5) | 0,08 | critère d'ajustement |

In [36]:
poids = {"semantique": 0.45, "famille": 0.20, "emotion": 0.15, "saison": 0.12, "intensite": 0.08}

candidats["score"] = (poids["semantique"] * candidats["s_semantique"]
                      + poids["famille"] * candidats["s_famille"]
                      + poids["emotion"] * candidats["s_emotion"]
                      + poids["saison"] * candidats["s_saison"]
                      + poids["intensite"] * candidats["s_intensite"])

Grâce au score, on peut maintenant classer et recommander les 3 parfums les plus adaptés au profil.
On trie les parfums du meilleur au moins bon score et on garde les 3 premiers.

In [37]:
colonnes_scores = ["s_semantique", "s_famille", "s_emotion", "s_saison", "s_intensite"]

top3 = candidats.sort_values(by="score", ascending=False).head(3)
top3[["name", "house", "score"] + colonnes_scores]

,name,house,score,s_semantique,s_famille,s_emotion,s_saison,s_intensite
732,Bucephalus No. XII,Armaf,0.963331,0.966102,1.0,0.857231,1.00,1.0
156,Sunday Cologne,Byredo,0.936375,0.974081,1.0,0.933587,0.65,1.0
1073,The One for Men Eau de Parfum,Dolce&Gabbana,0.929170,0.931463,1.0,0.733410,1.00,1.0


Ensuite, nous allons passé au score de couverture qui nous montre, critère par critère, le parfum recommandé qui couvre le profil de l’utilisateur. Ce sont les sous-scores du premier parfum : ils seront affichés sous forme de diagramme radar dans l’interface.

In [38]:
couverture = top3.iloc[0][colonnes_scores]
print("Parfum :", top3.iloc[0]["name"])
print(couverture)

Parfum : Bucephalus No. XII
s_semantique    0.966102
s_famille            1.0
s_emotion       0.857231
s_saison             1.0
s_intensite          1.0
Name: 732, dtype: object


Chaque recommandation est expliquée à partir de ses sous-scores : chaque sous-score assez élevé ajoute une phrase prédéfinie. On indique aussi les deux critères qui ont le plus contribué au score (poids × sous-score).

In [39]:
noms_criteres = {"semantique": "description", "famille": "famille olfactive",
                 "emotion": "émotions", "saison": "saison et moment", "intensite": "intensité"}

def expliquer(parfum):
    phrases = []
    if parfum["s_semantique"] >= 0.7:
        phrases.append(f"Votre description est très proche de ce parfum (similarité {parfum['cosinus_brut']:.2f}).")
    if parfum["s_famille"] == 1:
        phrases.append(f"Il appartient à la famille {parfum['family']} que vous avez choisie.")
    elif parfum["s_famille"] == 0.5:
        phrases.append(f"Sa famille secondaire ({parfum['secondary_family']}) fait partie de vos choix.")
    if parfum["s_emotion"] >= 0.7:
        phrases.append(f"Son caractère ({parfum['character'].replace('; ', ', ')}) correspond à l'ambiance recherchée.")
    if parfum["s_saison"] >= 0.7:
        phrases.append(f"Il se porte en {parfum['seasons'].replace('; ', ', ')}.")
    if parfum["s_intensite"] == 1:
        phrases.append(f"Son intensité ({parfum['concentration']}) correspond à vos attentes.")
    if not phrases:
        phrases.append("Il offre le meilleur compromis entre vos différents critères.")
    return " ".join(phrases)

def criteres_principaux(parfum):
    contributions = {c: poids[c] * parfum[f"s_{c}"] for c in poids}
    meilleurs = sorted(contributions, key=contributions.get, reverse=True)[:2]
    return [noms_criteres[c] for c in meilleurs]

for rang, (_, parfum) in enumerate(top3.iterrows(), start=1):
    print(f"{rang}. {parfum['name']} ({parfum['house']}) - score {parfum['score']:.2f}")
    print("   Critères principaux :", ", ".join(criteres_principaux(parfum)))
    print("   Explication :", expliquer(parfum))
    print()

1. Bucephalus No. XII (Armaf) - score 0.96
   Critères principaux : description, famille olfactive
   Explication : Votre description est très proche de ce parfum (similarité 0.61). Il appartient à la famille Spice Route que vous avez choisie. Son caractère (spicy, warm, earthy, fresh) correspond à l'ambiance recherchée. Il se porte en autumn, winter, spring. Son intensité (eau-de-parfum) correspond à vos attentes.

2. Sunday Cologne (Byredo) - score 0.94
   Critères principaux : description, famille olfactive
   Explication : Votre description est très proche de ce parfum (similarité 0.61). Il appartient à la famille Wood & Root que vous avez choisie. Son caractère (earthy, warm, spicy, fresh) correspond à l'ambiance recherchée. Son intensité (eau-de-parfum) correspond à vos attentes.

3. The One for Men Eau de Parfum (Dolce&Gabbana) - score 0.93
   Critères principaux : description, famille olfactive
   Explication : Votre description est très proche de ce parfum (similarité 0.59). I

On regroupe toutes les étapes dans une seule fonction, qui pourra être appelée par l’interface Streamlit. Si l’utilisateur ne répond pas à une question, le sous-score correspondant vaut 0,5 (valeur neutre).

In [40]:
def recommander_hybride(profil, n=3, poids=poids):
    # 1. Filtre des notes détestées
    masque = pd.Series(False, index=df.index)
    for note in profil.get("notes_detestees", []):
        masque = masque | df["notes"].str.contains(note, case=False)
    candidats = df[~masque].copy()

    # 2. Sous-scores (0,5 si la question est sans réponse)
    if profil.get("description"):
        cos = cosine_similarity(modele.encode([profil["description"]]), vecteurs_texte[candidats.index])[0]
        candidats["s_semantique"], candidats["cosinus_brut"] = normaliser(cos), cos
    else:
        candidats["s_semantique"], candidats["cosinus_brut"] = 0.5, 0.0

    if profil.get("familles"):
        principale = candidats["family"].isin(profil["familles"])
        secondaire = candidats["secondary_family"].isin(profil["familles"])
        candidats["s_famille"] = np.where(principale, 1.0, np.where(secondaire, 0.5, 0.0))
    else:
        candidats["s_famille"] = 0.5

    if profil.get("emotions"):
        cos = cosine_similarity(modele.encode([profil["emotions"]]), vecteurs_character[candidats.index])[0]
        candidats["s_emotion"] = normaliser(cos)
    else:
        candidats["s_emotion"] = 0.5

    saisons, moments = profil.get("saisons", []), profil.get("moments", [])
    s_saison = candidats[[f"season_{s}" for s in saisons]].mean(axis=1) if saisons else 0.5
    s_moment = candidats[[f"time_{m}" for m in moments]].mean(axis=1) if moments else 0.5
    candidats["s_saison"] = 0.7 * s_saison + 0.3 * s_moment

    if profil.get("intensite") is not None:
        candidats["s_intensite"] = (1 - (candidats["intensity"] - profil["intensite"]).abs() / 4).fillna(0.5)
    else:
        candidats["s_intensite"] = 0.5

    # 3. Score final
    candidats["score"] = sum(poids[c] * candidats[f"s_{c}"] for c in poids)

    # 4. Top n avec critères principaux et explications
    top = candidats.sort_values(by="score", ascending=False).head(n).copy()
    top["criteres_principaux"] = top.apply(criteres_principaux, axis=1)
    top["explication"] = top.apply(expliquer, axis=1)
    return top[["name", "house", "family", "score"] + colonnes_scores + ["criteres_principaux", "explication"]]

In [41]:
# On vérfie que la fonction fonctionne correctement avec le profil défini précédemment
recommander_hybride(profil)

,name,house,family,score,s_semantique,s_famille,s_emotion,s_saison,s_intensite,criteres_principaux,explication
732,Bucephalus No. XII,Armaf,Spice Route,0.963331,0.966102,1.0,0.857231,1.00,1.0,"[description, famille olfactive]",Votre description est très proche de ce parfum...
156,Sunday Cologne,Byredo,Wood & Root,0.936375,0.974081,1.0,0.933587,0.65,1.0,"[description, famille olfactive]",Votre description est très proche de ce parfum...
1073,The One for Men Eau de Parfum,Dolce&Gabbana,Spice Route,0.929170,0.931463,1.0,0.733410,1.00,1.0,"[description, famille olfactive]",Votre description est très proche de ce parfum...


Vérification : On vérifie que les parfums recommandés respectent bien les critères de l’utilisateur 

In [42]:
verification = top3[["name", "family", "character", "seasons", "times_of_day", "concentration", "notes"]].copy()
verification["contient_vanille"] = verification["notes"].str.contains("vanilla")
verification

,name,family,character,seasons,times_of_day,concentration,notes,contient_vanille
732,Bucephalus No. XII,Spice Route,spicy; warm; earthy; fresh,autumn; winter; spring,day; evening,eau-de-parfum,orange blossom; lemon; mint; spicy notes; pepp...,False
156,Sunday Cologne,Wood & Root,earthy; warm; spicy; fresh,spring; summer; autumn,day; evening,eau-de-parfum,bergamot; cardamom; anise; incense; geranium; ...,False
1073,The One for Men Eau de Parfum,Spice Route,spicy; warm; floral; fresh,autumn; winter; spring,day; evening,eau-de-parfum,basil; coriander; grapefruit; cardamom; ginger...,False


On retrouve bien le TOP 3 parfums de la fonction recommander_hybride()

### Conclusion

Le modèle hybride combine l'analyse sémantique de Sentence Transformers avec les critères structurés du dataset. Il exclut d’abord les parfums contenant une note détestée. Ensuite, il calcule 5 sous-scores (description, famille, émotions, saison, intensité), combinés en un score final pondéré. 
Et il recommande enfin les 3 meilleurs parfums, avec une explication pour chacun.

Comme on a pu le voir contrairement à la similarité seule, il comprend le sens de la demande et respecte le choix de l'utilsateur.

